Простейшая baseline-модель для оценки качества ML-модели:

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.neighbors import KNeighborsClassifier
import joblib
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('correct_dataset.csv')

Препроцессинг и разделение данных:

In [2]:
print("Исходный размер датасета:", len(df))
print("\nПропуски:")
print(df.isnull().sum())
df = df.dropna(subset=['text', 'category'])
print(f"\nПосле удаления пропусков: {len(df)}")

X = df['text'].values
y_orig = df['category'].values
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_orig)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print(f"\nРазмер обучающей выборки: {len(X_train)}")
print(f"Размер тестовой выборки: {len(X_test)}")

Исходный размер датасета: 1000

Пропуски:
text        12
category     0
priority     0
dtype: int64

После удаления пропусков: 988

Размер обучающей выборки: 790
Размер тестовой выборки: 198


Сырая векторизация текстов (без TF-IDF):

In [3]:
vectorizer = CountVectorizer(
    ngram_range=(1, 1),     
    stop_words=None,        
    max_features=25,                            
)
Xv_train = vectorizer.fit_transform(X_train)
Xv_test = vectorizer.transform(X_test)
Namesf = vectorizer.get_feature_names_out()
print("\nОбучаем модель по 25 самым частотным словам: ")
print(Namesf)


Обучаем модель по 25 самым частотным словам: 
['aaa' 'вернуть' 'возврат' 'доставка' 'зависнуть' 'заказ' 'курьер'
 'обменять' 'оплата' 'оплатить' 'оформить' 'ошибка' 'платёж' 'посылка'
 'почему' 'приложение' 'пришлый' 'программа' 'проходить' 'работать' 'сайт'
 'система' 'товар' 'удаваться' 'хотеть']


Обучение простейшей модели (в данном случае, knn с neighbors = 1) и анализ качества:

In [4]:
knn_model = KNeighborsClassifier(
    n_neighbors=1,           
    metric='euclidean',
)

knn_model.fit(Xv_train, y_train)
y_train_pred = knn_model.predict(Xv_train)
train_accuracy = accuracy_score(y_train, y_train_pred)
y_test_pred = knn_model.predict(Xv_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
test_f1_m = f1_score(y_test, y_test_pred, average='macro')
test_f1_w = f1_score(y_test, y_test_pred, average='weighted')
print(f"\nОбучение (train):")
print(f"  Accuracy: {train_accuracy:.4f}")
print(f"\nТестовая выборка (test):")
print(f"  Accuracy: {test_accuracy:.4f}")
print(f"  F1-macro: {test_f1_m:.4f}")
print(f"  F1-weighted: {test_f1_w:.4f}")


Обучение (train):
  Accuracy: 0.9418

Тестовая выборка (test):
  Accuracy: 0.8889
  F1-macro: 0.8881
  F1-weighted: 0.8879


Сохранение baseline:

In [5]:
joblib.dump(knn_model, 'baseline_knn1.pkl')
joblib.dump(vectorizer, 'countVectorizer.pkl')

['countVectorizer.pkl']